# POLAPP 칼·병 전용 객체 탐지 모델 미세조정

Open Images V7의 공식 바운딩 박스 주석과 TorchVision의 COCO 사전학습
`SSDLite320-MobileNetV3-Large`를 사용합니다.

- 탐지 클래스: `Bottle`, `Knife`
- 학습 결과: 3클래스 모델(`background`, `bottle`, `knife`)
- 평가: 클래스별 AP50, AP50:95, Precision, Recall, F1
- 배포 파일: ONNX, 임계값, 성능 지표, 모델 카드, PyTorch 체크포인트

**실행 전:** Colab 메뉴에서 `런타임 > 런타임 유형 변경 > T4 GPU`를 선택하고,
이전 설치 오류가 있었던 런타임이라면 `런타임 연결 해제 및 삭제` 후 새 런타임에서 실행하세요.
이 노트북은 FiftyOne, YOLOX, Pillow 재설치를 사용하지 않습니다.

In [ ]:
# 1. 필요한 패키지만 확인하여 설치
import importlib.util
import subprocess
import sys

packages = {
    "boto3": "boto3>=1.34,<2",
    "onnx": "onnx>=1.16,<2",
    "onnxruntime": "onnxruntime>=1.18,<2",
}
missing = [spec for module, spec in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
print("필수 패키지 준비 완료")

In [ ]:
# 2. 환경 및 학습 설정
from pathlib import Path
from collections import defaultdict
import copy
import csv
import json
import math
import random
import shutil
import subprocess
import time
import urllib.request

import numpy as np
import pandas as pd
import torch
import torchvision
from torch.utils.data import DataLoader, Dataset
from torchvision.io import ImageReadMode, read_image
from torchvision.ops import box_iou

SEED = 20260924
MAX_IMAGES_PER_CLASS = 700
TRAIN_RATIO = 0.75
VAL_RATIO = 0.15
BATCH_SIZE = 8
MAX_EPOCHS = 18
EARLY_STOPPING_PATIENCE = 4

ROOT = Path("/content/polapp_weapon_finetune")
DATA = ROOT / "data"
IMAGES = DATA / "images"
EXPORT = ROOT / "export"
for path in (DATA, IMAGES, EXPORT):
    path.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

assert torch.cuda.is_available(), "GPU 런타임이 아닙니다. T4 GPU를 선택하세요."
DEVICE = torch.device("cuda")
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("TorchVision:", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 3. Open Images V7 공식 주석에서 Bottle/Knife 표본 구성
BOXES_URL = "https://storage.googleapis.com/openimages/v5/validation-annotations-bbox.csv"
CLASSES_URL = "https://storage.googleapis.com/openimages/v7/oidv7-class-descriptions-boxable.csv"
DOWNLOADER_URL = "https://raw.githubusercontent.com/openimages/dataset/master/downloader.py"

boxes_csv = DATA / "validation-annotations-bbox.csv"
classes_csv = DATA / "oidv7-class-descriptions-boxable.csv"
downloader_py = DATA / "openimages_downloader.py"

def download_if_missing(url, destination):
    if not destination.exists():
        print("다운로드:", destination.name)
        urllib.request.urlretrieve(url, destination)

download_if_missing(BOXES_URL, boxes_csv)
download_if_missing(CLASSES_URL, classes_csv)
download_if_missing(DOWNLOADER_URL, downloader_py)

class_table = pd.read_csv(classes_csv, header=None, names=["LabelName", "DisplayName"])
wanted_names = {"Bottle", "Knife"}
target_rows = class_table[class_table["DisplayName"].isin(wanted_names)].copy()
found_names = set(target_rows["DisplayName"])
assert found_names == wanted_names, f"클래스 매핑 실패: {found_names}"
mid_to_name = dict(zip(target_rows["LabelName"], target_rows["DisplayName"]))
print("Open Images 클래스:", mid_to_name)

all_boxes = pd.read_csv(boxes_csv)
relevant = all_boxes[all_boxes["LabelName"].isin(mid_to_name)].copy()
relevant = relevant[
    (relevant["Confidence"] == 1)
    & (relevant["IsGroupOf"] == 0)
    & (relevant["IsDepiction"] == 0)
]
relevant["ClassName"] = relevant["LabelName"].map(mid_to_name)

rng = random.Random(SEED)
selected_ids = set()
for class_name in sorted(wanted_names):
    ids = sorted(relevant.loc[relevant["ClassName"] == class_name, "ImageID"].unique())
    rng.shuffle(ids)
    chosen = ids[:MAX_IMAGES_PER_CLASS]
    selected_ids.update(chosen)
    print(f"{class_name}: 전체 {len(ids):,}장 중 {len(chosen):,}장 선택")

annotations = relevant[relevant["ImageID"].isin(selected_ids)].copy()
image_ids = sorted(annotations["ImageID"].unique())
print(f"합계: 이미지 {len(image_ids):,}장, 객체 {len(annotations):,}개")
annotations.to_csv(DATA / "selected_annotations.csv", index=False)

In [ ]:
# 4. 공식 다운로더로 이미지를 내려받고 다중 라벨을 보존하여 분할
image_list = DATA / "image_ids.txt"
image_list.write_text("".join(f"validation/{image_id}\n" for image_id in image_ids), encoding="utf-8")

existing = {path.stem for path in IMAGES.glob("*.jpg")}
missing_ids = [image_id for image_id in image_ids if image_id not in existing]
if missing_ids:
    missing_list = DATA / "missing_image_ids.txt"
    missing_list.write_text(
        "".join(f"validation/{image_id}\n" for image_id in missing_ids),
        encoding="utf-8",
    )
    subprocess.check_call([
        sys.executable,
        str(downloader_py),
        str(missing_list),
        "--download_folder",
        str(IMAGES),
        "--num_processes",
        "16",
    ])

available_ids = [image_id for image_id in image_ids if (IMAGES / f"{image_id}.jpg").exists()]
assert len(available_ids) >= max(100, int(len(image_ids) * 0.95)), (
    f"이미지 다운로드가 불완전합니다: {len(available_ids)}/{len(image_ids)}"
)

labels_by_image = (
    annotations.groupby("ImageID")["ClassName"]
    .apply(lambda values: "+".join(sorted(set(values))))
    .to_dict()
)
grouped = defaultdict(list)
for image_id in available_ids:
    grouped[labels_by_image[image_id]].append(image_id)

train_ids, val_ids, test_ids = [], [], []
split_rng = random.Random(SEED)
for signature, ids in sorted(grouped.items()):
    split_rng.shuffle(ids)
    n = len(ids)
    train_end = max(1, int(n * TRAIN_RATIO))
    val_end = min(n - 1, train_end + max(1, int(n * VAL_RATIO))) if n >= 3 else train_end
    train_ids.extend(ids[:train_end])
    val_ids.extend(ids[train_end:val_end])
    test_ids.extend(ids[val_end:])
    print(signature, "->", len(ids[:train_end]), len(ids[train_end:val_end]), len(ids[val_end:]))

split_rng.shuffle(train_ids)
split_rng.shuffle(val_ids)
split_rng.shuffle(test_ids)
assert train_ids and val_ids and test_ids, "분할 중 하나가 비었습니다."

splits = {"train": train_ids, "val": val_ids, "test": test_ids}
(DATA / "splits.json").write_text(json.dumps(splits, indent=2), encoding="utf-8")
print("최종 분할:", {name: len(ids) for name, ids in splits.items()})

In [ ]:
# 5. 객체 탐지 데이터셋
CLASS_TO_ID = {"Bottle": 1, "Knife": 2}
ID_TO_CLASS = {0: "background", 1: "bottle", 2: "knife"}

annotations_by_image = {
    image_id: frame.reset_index(drop=True)
    for image_id, frame in annotations.groupby("ImageID")
}

class WeaponDataset(Dataset):
    def __init__(self, ids, train=False):
        self.ids = list(ids)
        self.train = train

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, index):
        image_id = self.ids[index]
        image = read_image(str(IMAGES / f"{image_id}.jpg"), mode=ImageReadMode.RGB).float() / 255.0
        _, height, width = image.shape
        rows = annotations_by_image[image_id]

        boxes = torch.tensor([
            [
                row.XMin * width,
                row.YMin * height,
                row.XMax * width,
                row.YMax * height,
            ]
            for row in rows.itertuples()
        ], dtype=torch.float32)
        labels = torch.tensor(
            [CLASS_TO_ID[row.ClassName] for row in rows.itertuples()],
            dtype=torch.int64,
        )

        if self.train and random.random() < 0.5:
            image = torch.flip(image, dims=[2])
            old_x1 = boxes[:, 0].clone()
            old_x2 = boxes[:, 2].clone()
            boxes[:, 0] = width - old_x2
            boxes[:, 2] = width - old_x1

        if self.train:
            brightness = random.uniform(0.85, 1.15)
            contrast = random.uniform(0.85, 1.15)
            mean = image.mean(dim=(1, 2), keepdim=True)
            image = ((image - mean) * contrast + mean) * brightness
            image = image.clamp(0, 1)

        area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
        target = {
            "boxes": boxes,
            "labels": labels,
            "area": area,
            "iscrowd": torch.zeros(len(boxes), dtype=torch.int64),
            "image_id": torch.tensor([int(image_id[:8], 16)], dtype=torch.int64),
        }
        return image, target, image_id

def collate_fn(batch):
    images, targets, image_ids = zip(*batch)
    return list(images), list(targets), list(image_ids)

train_loader = DataLoader(
    WeaponDataset(train_ids, train=True),
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True,  # BatchNorm 오류를 막기 위해 마지막 1장짜리 배치를 제외
    num_workers=2,
    pin_memory=True,
    collate_fn=collate_fn,
)
val_loader = DataLoader(
    WeaponDataset(val_ids),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate_fn,
)
test_loader = DataLoader(
    WeaponDataset(test_ids),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    collate_fn=collate_fn,
)
print("데이터 로더 준비 완료")

In [ ]:
# 6. COCO 사전학습 SSDLite를 3클래스 전용 모델로 전환
from torchvision.models.detection import (
    SSDLite320_MobileNet_V3_Large_Weights,
    ssdlite320_mobilenet_v3_large,
)

coco_model = ssdlite320_mobilenet_v3_large(
    weights=SSDLite320_MobileNet_V3_Large_Weights.DEFAULT
)
model = ssdlite320_mobilenet_v3_large(
    weights=None,
    weights_backbone=None,
    num_classes=3,
    score_thresh=0.01,
    nms_thresh=0.45,
    detections_per_img=100,
)

source_state = coco_model.state_dict()
target_state = model.state_dict()
compatible_state = {
    key: value
    for key, value in source_state.items()
    if key in target_state and value.shape == target_state[key].shape
}
load_result = model.load_state_dict(compatible_state, strict=False)
print(f"전이된 텐서: {len(compatible_state):,}개")
print(f"새로 학습할 출력 텐서: {len(load_result.missing_keys):,}개")
del coco_model, source_state, target_state, compatible_state
torch.cuda.empty_cache()

model.to(DEVICE)
backbone_params = list(model.backbone.parameters())
backbone_ids = {id(parameter) for parameter in backbone_params}
head_params = [parameter for parameter in model.parameters() if id(parameter) not in backbone_ids]
optimizer = torch.optim.SGD(
    [
        {"params": backbone_params, "lr": 5e-4},
        {"params": head_params, "lr": 2e-3},
    ],
    momentum=0.9,
    weight_decay=5e-4,
    nesterov=True,
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
print("모델 준비 완료")

In [ ]:
# 7. 평가 함수: 이미지별 일대일 IoU 매칭으로 AP와 P/R/F1 계산
@torch.inference_mode()
def collect_predictions(loader):
    model.eval()
    records = []
    for images, targets, image_ids in loader:
        device_images = [image.to(DEVICE, non_blocking=True) for image in images]
        outputs = model(device_images)
        for image_id, target, output in zip(image_ids, targets, outputs):
            records.append({
                "image_id": image_id,
                "target_boxes": target["boxes"].cpu(),
                "target_labels": target["labels"].cpu(),
                "boxes": output["boxes"].cpu(),
                "labels": output["labels"].cpu(),
                "scores": output["scores"].cpu(),
            })
    return records

def class_match(records, class_id, iou_threshold=0.5, score_threshold=0.0):
    ground_truth = {}
    total_gt = 0
    detections = []
    for record in records:
        gt_mask = record["target_labels"] == class_id
        gt_boxes = record["target_boxes"][gt_mask]
        ground_truth[record["image_id"]] = {
            "boxes": gt_boxes,
            "used": torch.zeros(len(gt_boxes), dtype=torch.bool),
        }
        total_gt += len(gt_boxes)

        pred_mask = (record["labels"] == class_id) & (record["scores"] >= score_threshold)
        for box, score in zip(record["boxes"][pred_mask], record["scores"][pred_mask]):
            detections.append((float(score), record["image_id"], box))

    detections.sort(key=lambda item: item[0], reverse=True)
    tp, fp, scores = [], [], []
    for score, image_id, box in detections:
        entry = ground_truth[image_id]
        matched = False
        if len(entry["boxes"]):
            ious = box_iou(box.unsqueeze(0), entry["boxes"])[0]
            best_iou, best_index = ious.max(dim=0)
            if best_iou >= iou_threshold and not entry["used"][best_index]:
                entry["used"][best_index] = True
                matched = True
        tp.append(1.0 if matched else 0.0)
        fp.append(0.0 if matched else 1.0)
        scores.append(score)
    return np.asarray(tp), np.asarray(fp), np.asarray(scores), total_gt

def average_precision(records, class_id, iou_threshold):
    tp, fp, _, total_gt = class_match(records, class_id, iou_threshold, 0.0)
    if total_gt == 0 or len(tp) == 0:
        return 0.0
    tp = np.cumsum(tp)
    fp = np.cumsum(fp)
    recall = tp / max(total_gt, 1)
    precision = tp / np.maximum(tp + fp, 1e-9)
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([0.0], precision, [0.0]))
    for index in range(len(mpre) - 2, -1, -1):
        mpre[index] = max(mpre[index], mpre[index + 1])
    changing = np.where(mrec[1:] != mrec[:-1])[0]
    return float(np.sum((mrec[changing + 1] - mrec[changing]) * mpre[changing + 1]))

def precision_recall_f1(records, class_id, threshold, iou_threshold=0.5):
    tp, fp, _, total_gt = class_match(records, class_id, iou_threshold, threshold)
    true_positive = float(tp.sum())
    false_positive = float(fp.sum())
    false_negative = float(max(total_gt - true_positive, 0))
    precision = true_positive / max(true_positive + false_positive, 1e-9)
    recall = true_positive / max(true_positive + false_negative, 1e-9)
    f1 = 2 * precision * recall / max(precision + recall, 1e-9)
    return {"precision": precision, "recall": recall, "f1": f1}

def map50(records):
    return float(np.mean([average_precision(records, class_id, 0.5) for class_id in (1, 2)]))

print("평가 함수 준비 완료")

In [ ]:
# 8. 미세조정: 검증 mAP50 기준 최적 체크포인트 저장
best_map50 = -1.0
best_state = None
history = []
stale_checks = 0

for epoch in range(1, MAX_EPOCHS + 1):
    started = time.time()
    model.train()
    running_loss = 0.0
    batches = 0
    for images, targets, _ in train_loader:
        images = [image.to(DEVICE, non_blocking=True) for image in images]
        targets = [
            {key: value.to(DEVICE, non_blocking=True) for key, value in target.items()}
            for target in targets
        ]
        optimizer.zero_grad(set_to_none=True)
        loss_dict = model(images, targets)
        loss = sum(loss_dict.values())
        if not torch.isfinite(loss):
            raise RuntimeError(f"비정상 손실값: {loss_dict}")
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
        optimizer.step()
        running_loss += float(loss.detach())
        batches += 1
    scheduler.step()

    row = {
        "epoch": epoch,
        "train_loss": running_loss / max(batches, 1),
        "seconds": time.time() - started,
        "lr_backbone": optimizer.param_groups[0]["lr"],
        "lr_head": optimizer.param_groups[1]["lr"],
    }

    should_evaluate = epoch == 1 or epoch % 2 == 0 or epoch == MAX_EPOCHS
    if should_evaluate:
        validation_records = collect_predictions(val_loader)
        current_map50 = map50(validation_records)
        row["val_map50"] = current_map50
        if current_map50 > best_map50 + 1e-4:
            best_map50 = current_map50
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            stale_checks = 0
        else:
            stale_checks += 1
        print(
            f"Epoch {epoch:02d} | loss {row['train_loss']:.4f} | "
            f"val mAP50 {current_map50:.4f} | {row['seconds']:.0f}s"
        )
        if stale_checks >= EARLY_STOPPING_PATIENCE:
            print("검증 성능이 개선되지 않아 조기 종료합니다.")
            history.append(row)
            break
    else:
        print(f"Epoch {epoch:02d} | loss {row['train_loss']:.4f} | {row['seconds']:.0f}s")
    history.append(row)

assert best_state is not None, "최적 체크포인트를 만들지 못했습니다."
model.load_state_dict(best_state)
model.to(DEVICE).eval()
torch.save(best_state, EXPORT / "weapon_detector_finetuned.pth")
(EXPORT / "training_history.json").write_text(
    json.dumps(history, indent=2), encoding="utf-8"
)
print("최적 검증 mAP50:", round(best_map50, 4))

In [ ]:
# 9. 검증 세트로 클래스별 신뢰도 임계값을 보정하고 테스트 세트 최종 평가
validation_records = collect_predictions(val_loader)
test_records = collect_predictions(test_loader)
thresholds = {}
metrics = {
    "dataset": "Open Images V7 validation subset with deterministic image-level split",
    "split_sizes": {name: len(ids) for name, ids in splits.items()},
    "best_validation_map50": best_map50,
    "classes": {},
}

for class_id in (1, 2):
    class_name = ID_TO_CLASS[class_id]
    candidates = np.arange(0.10, 0.91, 0.05)
    scored = [
        (float(threshold), precision_recall_f1(validation_records, class_id, float(threshold)))
        for threshold in candidates
    ]
    best_threshold, best_validation = max(scored, key=lambda item: item[1]["f1"])
    thresholds[class_name] = best_threshold
    final_prf = precision_recall_f1(test_records, class_id, best_threshold)
    ap50 = average_precision(test_records, class_id, 0.5)
    ap_values = [
        average_precision(test_records, class_id, float(iou))
        for iou in np.arange(0.50, 0.96, 0.05)
    ]
    metrics["classes"][class_name] = {
        "threshold": best_threshold,
        "validation_f1": best_validation["f1"],
        "test_ap50": ap50,
        "test_ap50_95": float(np.mean(ap_values)),
        **{f"test_{key}": value for key, value in final_prf.items()},
    }

for metric_name in ("test_ap50", "test_ap50_95", "test_precision", "test_recall", "test_f1"):
    metrics[f"macro_{metric_name}"] = float(np.mean([
        class_metrics[metric_name] for class_metrics in metrics["classes"].values()
    ]))

print(json.dumps(metrics, indent=2, ensure_ascii=False))
(EXPORT / "thresholds.json").write_text(
    json.dumps(thresholds, indent=2), encoding="utf-8"
)
(EXPORT / "metrics.json").write_text(
    json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf-8"
)

In [ ]:
# 10. 테스트 이미지에서 탐지 결과 확인
import matplotlib.pyplot as plt
from torchvision.utils import draw_bounding_boxes

sample_dataset = WeaponDataset(test_ids)
figure, axes = plt.subplots(2, 2, figsize=(12, 10))
sample_indices = np.linspace(0, len(sample_dataset) - 1, 4, dtype=int)
model.eval()
for axis, sample_index in zip(axes.flat, sample_indices):
    image, _, image_id = sample_dataset[int(sample_index)]
    with torch.inference_mode():
        output = model([image.to(DEVICE)])[0]
    keep = torch.tensor([
        float(score) >= thresholds[ID_TO_CLASS[int(label)]]
        for score, label in zip(output["scores"].cpu(), output["labels"].cpu())
    ], dtype=torch.bool)
    boxes = output["boxes"].cpu()[keep]
    labels = [
        f"{ID_TO_CLASS[int(label)]} {float(score):.2f}"
        for label, score in zip(output["labels"].cpu()[keep], output["scores"].cpu()[keep])
    ]
    canvas = draw_bounding_boxes(
        (image * 255).byte(), boxes, labels=labels, colors="red", width=3
    )
    axis.imshow(canvas.permute(1, 2, 0))
    axis.set_title(image_id)
    axis.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# 11. ONNX 내보내기, 실행 검증, 최종 번들 다운로드
import onnx
import onnxruntime as ort

class ExportWrapper(torch.nn.Module):
    def __init__(self, detector):
        super().__init__()
        self.detector = detector

    def forward(self, image):
        output = self.detector([image[0]])[0]
        return output["boxes"], output["scores"], output["labels"]

cpu_model = copy.deepcopy(model).cpu().eval()
wrapper = ExportWrapper(cpu_model).eval()
dummy = torch.zeros(1, 3, 320, 320, dtype=torch.float32)
onnx_path = EXPORT / "weapon_detector_finetuned.onnx"
torch.onnx.export(
    wrapper,
    dummy,
    str(onnx_path),
    opset_version=17,
    input_names=["image"],
    output_names=["boxes", "scores", "labels"],
    dynamic_axes={
        "boxes": {0: "num_detections"},
        "scores": {0: "num_detections"},
        "labels": {0: "num_detections"},
    },
    do_constant_folding=True,
    dynamo=False,
)
onnx_model = onnx.load(str(onnx_path))
onnx.checker.check_model(onnx_model)
session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
ort_outputs = session.run(None, {"image": np.zeros((1, 3, 320, 320), dtype=np.float32)})
print("ONNX 검증 완료:", [output.shape for output in ort_outputs])

(EXPORT / "classes.json").write_text(
    json.dumps({str(key): value for key, value in ID_TO_CLASS.items()}, indent=2),
    encoding="utf-8",
)
model_card = f'''# POLAPP Weapon Detector

Architecture: SSDLite320-MobileNetV3-Large
Initialization: TorchVision COCO pretrained weights, compatible layers transferred
Fine-tuning data: Open Images V7 Bottle/Knife bounding boxes
Classes: background, bottle, knife
Best validation mAP50: {best_map50:.4f}
Test macro AP50: {metrics['macro_test_ap50']:.4f}
Test macro AP50:95: {metrics['macro_test_ap50_95']:.4f}

This model is a field decision-support prototype. A bottle detection alone must not be
treated as a weapon determination. Before operational use, evaluate with Korean police
body-camera/phone-camera footage, including soju bottles, kitchen knives, occlusion,
night scenes, motion blur, and hard negative objects.

Open Images annotations are CC BY 4.0. Image licenses and attribution requirements must
be checked before redistributing training images.
'''
(EXPORT / "model_card.md").write_text(model_card, encoding="utf-8")

bundle_base = ROOT / "polapp_weapon_detector_finetuned_bundle"
archive = shutil.make_archive(str(bundle_base), "zip", EXPORT)
print("최종 번들:", archive)
print("파일:", [path.name for path in sorted(EXPORT.iterdir())])

from google.colab import files
files.download(archive)